# pg_gpu: A Tour of GPU-Accelerated Population Genetics

This notebook demonstrates the major features of **pg_gpu**, a GPU-accelerated population genetics statistics library. We use phased haplotype data from 100 *Anopheles gambiae* individuals (a 4 Mb region of the X chromosome) from the [Ag1000G project](https://www.malariagen.net/data/ag1000g-phase3).

**Sections covered:**
1. Loading VCF data and the `HaplotypeMatrix`
2. GPU acceleration
3. Site frequency spectrum (SFS)
4. Diversity statistics
5. Divergence and differentiation
6. Windowed genome scans
7. Linkage disequilibrium
8. PCA and population structure
9. Selection scans

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

import pg_gpu
from pg_gpu import (
    HaplotypeMatrix,
    diversity,
    divergence,
    sfs,
    windowed_analysis,
    selection,
    decomposition,
    plotting,
)

sns.set_style("whitegrid")
print(f"pg_gpu version: {pg_gpu.__version__}")

## 1. Loading Data

The central data structure is `HaplotypeMatrix`. Load phased VCF data directly with `from_vcf()`.

In [ ]:
vcf_path = "data/gamb.X.8-12Mb.n100.derived.vcf.gz"

t0 = time.time()
h = HaplotypeMatrix.from_vcf(vcf_path)
load_time = time.time() - t0

print(f"Loaded in {load_time:.1f}s")
print(f"Shape: {h.shape}  ({h.num_haplotypes} haplotypes x {h.num_variants} variants)")
print(f"Genomic span: {h.positions[0]:,} - {h.positions[-1]:,} bp")
print(f"Device: {h.device}")

### Defining populations

We can assign samples to named populations via `sample_sets`. Here we split the 100 diploid individuals (200 haplotypes) into two arbitrary groups to demonstrate two-population statistics.

In [ ]:
# Split haplotypes: first 50 diploids -> pop_A, next 50 -> pop_B
h.sample_sets = {
    "pop_A": list(range(0, 100)),    # haplotypes 0-99
    "pop_B": list(range(100, 200)),  # haplotypes 100-199
}
print(f"pop_A: {len(h.sample_sets['pop_A'])} haplotypes")
print(f"pop_B: {len(h.sample_sets['pop_B'])} haplotypes")

## 2. GPU Acceleration

Transfer data to the GPU with a single call. All downstream computations then run on the GPU automatically.

In [ ]:
h.transfer_to_gpu()
print(f"Device: {h.device}")

## 3. Site Frequency Spectrum

Compute the unfolded and folded SFS, plus the joint SFS between two populations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Unfolded SFS (all samples)
s_unfolded = sfs.sfs(h)
plotting.plot_sfs(s_unfolded, ax=axes[0])
axes[0].set_title("Unfolded SFS")

# Folded SFS (all samples)
s_folded = sfs.sfs_folded(h)
plotting.plot_sfs(s_folded, ax=axes[1], folded=True)
axes[1].set_title("Folded SFS")

# Joint SFS between pop_A and pop_B
s_joint = sfs.joint_sfs(h, pop1="pop_A", pop2="pop_B")
plotting.plot_joint_sfs(s_joint, ax=axes[2])
axes[2].set_title("Joint SFS (pop_A vs pop_B)")

plt.tight_layout()
plt.show()

## 4. Diversity Statistics

pg_gpu computes standard within-population diversity statistics, all fully vectorized on the GPU.

In [ ]:
for pop_name in ["pop_A", "pop_B"]:
    pi_val = diversity.pi(h, population=pop_name)
    tw_val = diversity.theta_w(h, population=pop_name)
    td_val = diversity.tajimas_d(h, population=pop_name)
    ss_val = diversity.segregating_sites(h, population=pop_name)
    sc_val = diversity.singleton_count(h, population=pop_name)

    print(f"--- {pop_name} ---")
    print(f"  pi            = {pi_val:.6f}")
    print(f"  theta_W       = {tw_val:.6f}")
    print(f"  Tajima's D    = {td_val:.4f}")
    print(f"  S (seg sites) = {ss_val:,}")
    print(f"  singletons    = {sc_val:,}")
    print()

## 5. Divergence and Differentiation

Compute between-population statistics: $F_{ST}$ (multiple estimators), $D_{xy}$, and $D_a$.

In [ ]:
# FST with different estimators
for method in ["hudson", "weir_cockerham", "nei"]:
    fst_val = divergence.fst(h, pop1="pop_A", pop2="pop_B", method=method)
    print(f"FST ({method:>15s}) = {fst_val:.6f}")

print()

# Dxy and Da
dxy_val = divergence.dxy(h, pop1="pop_A", pop2="pop_B")
da_val = divergence.da(h, pop1="pop_A", pop2="pop_B")
print(f"Dxy = {dxy_val:.6f}")
print(f"Da  = {da_val:.6f}")

## 6. Windowed Genome Scans

Scan the chromosome in non-overlapping windows to visualize how statistics vary along the genome. The fused CUDA kernel computes all statistics across all windows in a single GPU pass.

In [ ]:
t0 = time.time()
results = windowed_analysis(
    h,
    window_size=100_000,
    statistics=["pi", "theta_w", "tajimas_d"],
)
elapsed = time.time() - t0
print(f"Windowed analysis: {len(results)} windows in {elapsed:.1f}s")
results.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

colors = sns.color_palette("Set2", 3)
stats = ["pi", "theta_w", "tajimas_d"]
labels = [r"$\pi$", r"$\theta_W$", "Tajima's D"]

for ax, stat, label, color in zip(axes, stats, labels, colors):
    plotting.plot_windowed(
        results["window_start"].values,
        results[stat].values,
        ax=ax,
        stat_name=label,
        color=color,
    )

axes[-1].set_xlabel("Position on X chromosome (Mb)")
fig.suptitle("Windowed diversity statistics (100 kb windows)", y=1.01)
plt.tight_layout()
plt.show()

### Windowed $F_{ST}$

We can also scan divergence statistics in windows.

In [ ]:
results_fst = windowed_analysis(
    h,
    window_size=100_000,
    statistics=["fst"],
    populations=["pop_A", "pop_B"],
)

fig, ax = plt.subplots(figsize=(12, 3))
plotting.plot_windowed(
    results_fst["window_start"].values,
    results_fst["fst"].values,
    ax=ax,
    stat_name=r"$F_{ST}$ (Hudson)",
    color=sns.color_palette("Set1")[0],
)
ax.set_xlabel("Position on X chromosome (Mb)")
ax.set_title(r"Windowed $F_{ST}$ between pop_A and pop_B")
plt.tight_layout()
plt.show()

## 7. Linkage Disequilibrium

Compute pairwise $r^2$ and visualize LD structure. We use a small genomic region to keep computation tractable.

In [ ]:
# Extract a small region for LD visualization
import cupy as cp
region_start, region_end = 10_000_000, 10_010_000
pos_gpu = h.positions if isinstance(h.positions, cp.ndarray) else cp.asarray(h.positions)
sel_idx = cp.where((pos_gpu >= region_start) & (pos_gpu < region_end))[0]
h_region = h.get_subset(sel_idx)
print(f"Region: {region_start:,}-{region_end:,} bp, {h_region.num_variants} variants")

# Compute pairwise r^2
r2_matrix = h_region.pairwise_r2().get()

fig, ax = plt.subplots(figsize=(7, 6))
pos_region = h_region.positions.get() if hasattr(h_region.positions, 'get') else h_region.positions
plotting.plot_pairwise_ld(r2_matrix, ax=ax, positions=pos_region)
ax.set_title(f"Pairwise LD ($r^2$), X:{region_start/1e6:.1f}-{region_end/1e3:.0f}kb")
plt.tight_layout()
plt.show()

### LD Decay

Visualize how LD decays with physical distance.

In [ ]:
# Compute distances and r^2 values for all pairs in the region
r2_np = r2_matrix.get() if hasattr(r2_matrix, 'get') else np.asarray(r2_matrix)
positions_np = h_region.positions.get() if hasattr(h_region.positions, 'get') else np.asarray(h_region.positions)
n_snps = len(positions_np)

# Extract upper triangle
row_idx, col_idx = np.triu_indices(n_snps, k=1)
distances = positions_np[col_idx] - positions_np[row_idx]
r2_vals = r2_np[row_idx, col_idx]

fig, ax = plt.subplots(figsize=(8, 4))
plotting.plot_ld_decay(distances, r2_vals, ax=ax, bins=30)
ax.set_title("LD decay with physical distance")
plt.tight_layout()
plt.show()

## 8. PCA and Population Structure

Run PCA on the GPU to visualize population structure among samples.

In [ ]:
t0 = time.time()
coords, explained_var = decomposition.pca(h, n_components=4, scaler="patterson")
pca_time = time.time() - t0

print(f"PCA computed in {pca_time:.1f}s")
print(f"Variance explained: " + ", ".join(f"PC{i+1}={v:.1%}" for i, v in enumerate(explained_var)))

# Label each haplotype by population
labels = np.array(["pop_A"] * 100 + ["pop_B"] * 100)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plotting.plot_pca(coords, ax=axes[0], pc_x=0, pc_y=1,
                  explained_variance=explained_var, labels=labels)
axes[0].set_title("PC1 vs PC2")

plotting.plot_pca(coords, ax=axes[1], pc_x=0, pc_y=2,
                  explained_variance=explained_var, labels=labels)
axes[1].set_title("PC1 vs PC3")

plt.tight_layout()
plt.show()

## 9. Selection Scans

### Garud's H statistics

Haplotype homozygosity statistics that detect recent positive selection. We compute them in sliding windows across the chromosome.

In [ ]:
# Garud's H in sliding windows (1000-SNP windows, step 500)
h1, h12, h123, h2_h1 = selection.moving_garud_h(h, size=1000, step=500)

# Get window center positions for x-axis
positions_cpu = h.positions.get() if hasattr(h.positions, 'get') else np.asarray(h.positions)
n_variants = h.num_variants
centers = []
for w_start in range(0, n_variants - 1000 + 1, 500):
    center_pos = (positions_cpu[w_start] + positions_cpu[w_start + 999]) / 2
    centers.append(center_pos)
centers = np.array(centers)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
colors = sns.color_palette("Dark2", 3)

for ax, vals, label, color in zip(
    axes, [h1, h12, h2_h1], ["H1", "H12", "H2/H1"], colors
):
    plotting.plot_windowed(centers, vals, ax=ax, stat_name=label, color=color)

axes[-1].set_xlabel("Position on X chromosome (Mb)")
fig.suptitle("Garud's H statistics (1000-SNP windows)", y=1.01)
plt.tight_layout()
plt.show()

### nSL (number of segregating sites by length)

A haplotype-based statistic for detecting selection that is robust to variation in recombination rate. Computed per-variant and then standardized by derived allele count.

In [ ]:
# nSL is computed per-variant across the full dataset
t0 = time.time()
nsl_scores = selection.nsl(h)
nsl_time = time.time() - t0
print(f"nSL computed on {h.num_variants:,} variants in {nsl_time:.1f}s")

# Standardize by derived allele count
dac, _ = sfs._derived_allele_counts(h)
dac_np = dac.get() if hasattr(dac, 'get') else np.asarray(dac)
nsl_std, bins_used = selection.standardize_by_allele_count(nsl_scores, dac_np)

pos_nsl = h.positions.get() if hasattr(h.positions, 'get') else np.asarray(h.positions)
valid = ~np.isnan(nsl_std)

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(pos_nsl[valid] / 1e6, nsl_std[valid], s=1, alpha=0.3, c="steelblue", rasterized=True)
ax.axhline(2, color="red", linestyle="--", linewidth=0.8, label="nSL = 2")
ax.axhline(-2, color="red", linestyle="--", linewidth=0.8, label="nSL = -2")
ax.set_xlabel("Position (Mb)")
ax.set_ylabel("Standardized nSL")
ax.set_title("nSL scan, X:8-12 Mb")
ax.legend(loc="upper right")
sns.despine(ax=ax)
plt.tight_layout()
plt.show()

## Summary

This tour demonstrated the major features of pg_gpu:

| Feature | Module |
|---|---|
| Data loading (VCF) | `HaplotypeMatrix.from_vcf()` |
| GPU acceleration | `transfer_to_gpu()` / `transfer_to_cpu()` |
| Site frequency spectra | `pg_gpu.sfs` |
| Diversity ($\pi$, $\theta_W$, Tajima's D, ...) | `pg_gpu.diversity` |
| Divergence ($F_{ST}$, $D_{xy}$, $D_a$) | `pg_gpu.divergence` |
| Windowed genome scans | `pg_gpu.windowed_analysis` |
| Linkage disequilibrium ($r^2$, LD decay) | `HaplotypeMatrix.pairwise_r2()`, `pg_gpu.ld_statistics` |
| PCA | `pg_gpu.decomposition` |
| Selection scans (Garud's H, nSL, iHS) | `pg_gpu.selection` |

Additional features not shown here include admixture statistics (`pg_gpu.admixture`), pairwise distance matrices (`pg_gpu.decomposition.pairwise_distance`), and flexible missing data handling (three modes: `'include'`, `'exclude'`, `'pairwise'`).